In [ ]:
import pandas as pd

# dataset
df = pd.read_csv("online_retail.csv")

df.head()


In [ ]:
df.info()
df.shape

In [ ]:
# Data Cleaning

# drop missing customerID
df_clean = df.dropna(subset=["CustomerID"])

# drop negative quantity
df_clean = df_clean[df_clean["Quantity"] > 0]

# drop negative price
df_clean = df_clean[df_clean["UnitPrice"] > 0]

# conversion InvoiceDate to datetime
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

# add revenue coilumn
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]


df_clean.shape

In [ ]:
# Customer level analysis

customer_df = df_clean.groupby("CustomerID").agg(
    {
        "InvoiceNo": "nunique",  # number of unique invoices
        "Revenue": "sum",        # total revenue
        "InvoiceDate": ["min", "max"],    # last purchase date
    }
)

customer_df.columns = ["TotalOrder", "TotalRevenue", "FirstPurchase", "LastPurchase"]
customer_df.reset_index(inplace=True)

customer_df.head()

In [ ]:
# Business Matrix

# Customer lifetime in days
customer_df["Customer_lifetime"] = (customer_df["LastPurchase"] - customer_df["FirstPurchase"]).dt.days


# Average order value
customer_df["AvgOrderValue"] = round(customer_df["TotalRevenue"] / customer_df["TotalOrder"], 2)


In [ ]:
# Fixing lifetime
customer_df["Customer_lifetime"] = customer_df["Customer_lifetime"].apply(lambda x: 1 if x == 0 else x)


# Purchase frequency
customer_df['PurchaseFrequency'] = (
    customer_df['TotalOrder'] / customer_df['Customer_lifetime']
) * 365

# Customer Lifetime Value (CLV)
customer_df['CLV'] = (
    customer_df['AvgOrderValue'] *
    customer_df['PurchaseFrequency'] *
    customer_df['Customer_lifetime'] / 365
)

customer_df[['TotalRevenue', 'CLV']].describe()

In [ ]:
# CLV Segmentation
customer_df['CLV_Segment'] = pd.qcut(
    customer_df['CLV'],
    q=3,
    labels=['Low Value', 'Medium Value', 'High Value']
)

customer_df['CLV_Segment'].value_counts()

In [ ]:
# Revenue distribution
segment_revenue = customer_df.groupby('CLV_Segment')['TotalRevenue'].sum()
segment_revenue

In [ ]:
# Data save

customer_df.to_csv("customer_clv.csv", index=False)


In [ ]:
# Churn Prediction

# Ensure InvoiceDate is datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Reference date = last date in dataset
reference_date = df['InvoiceDate'].max()

# Last purchase per customer
last_purchase = (
    df.groupby('CustomerID')['InvoiceDate']
    .max()
    .reset_index()
)

# Days since last purchase
last_purchase['DaysSinceLastPurchase'] = (
    reference_date - last_purchase['InvoiceDate']
).dt.days

# Merge with customer_df
customer_df = customer_df.merge(
    last_purchase[['CustomerID', 'DaysSinceLastPurchase']],
    on='CustomerID',
    how='left'
)

# Create churn label (90-day rule)
customer_df['Churn'] = customer_df['DaysSinceLastPurchase'].apply(
    lambda x: 1 if x > 90 else 0
)

customer_df['Churn'].value_counts()


In [ ]:
# Data Preparation for Modeling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = [
    'DaysSinceLastPurchase',
    'TotalOrder',
    'AvgOrderValue',
    'PurchaseFrequency',
    'Customer_lifetime'
]

X = customer_df[features]
y = customer_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Model Training and Evaluation

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))


In [ ]:
# Predict churn probabilities for all customers

customer_df['Churn_Probability'] = model.predict_proba(
    scaler.transform(customer_df[features])
)[:, 1]


In [ ]:
# Retention Priority Scoring

# Normalize CLV
customer_df['CLV_Normalized'] = (
    customer_df['CLV'] / customer_df['CLV'].max()
)

# Retention Priority Score
customer_df['Retention_Priority_Score'] = (
    customer_df['CLV_Normalized'] * customer_df['Churn_Probability']
)
# Rank customers by retention priority
customer_df['Retention_Priority_Rank'] = customer_df['Retention_Priority_Score'].rank(ascending=False)
print(customer_df[['CustomerID', 'CLV', 'Churn_Probability', 'Retention_Priority_Score', 'Retention_Priority_Rank']].head(10))


In [ ]:
customer_df.to_csv("customer_clv_ai.csv", index=False)
